# Solving combinatorial problems with QAOA

This notebook demonstrates the process of solving a combinatorial problem using the QUBO notation and the QAOA VQA.

In this example we will be tackling the traveling salesman problem (TSP) which aims at finding the lowest cost cycle in a graph while visiting each node once.

In [14]:
import networkx # used here to generate the graph
import numpy as np
import numpy.typing as npt
import random

First we will be generating the graph.

In [15]:
G = networkx.fast_gnp_random_graph(3,1)
cost_list = []
for (u,v,w) in G.edges(data=True):
    w['weight'] = random.randint(3,10)
    cost_list.append(([u,v],w['weight']))
    cost_list.append([[v,u], w['weight'] + random.randint(-3,10)])
print(G)
print()
print(cost_list)

Graph with 3 nodes and 3 edges

[([0, 1], 3), [[1, 0], 2], ([0, 2], 4), [[2, 0], 9], ([1, 2], 5), [[2, 1], 11]]


Now that we have our graph with random weights we need to get the adjacency matrix of this graph.

This is a matrix that represent all of the weights between each nodes.

In [16]:
def generate_adjacency_matrix(list_of_cost : list[tuple[list[int], int]], nb_cities: int):
    """Example of list of cost = [[[0,1],23], [[1,2], 52], [[0,2], 12]] """
    w_ij = np.zeros((nb_cities, nb_cities), dtype=int)
    for i in list_of_cost:
        cost = i[1]
        w_ij[i[0][0], i[0][1]] = cost
    return w_ij
adj_matrix = generate_adjacency_matrix(cost_list,3)
print(adj_matrix) 

[[ 0  3  4]
 [ 2  0  5]
 [ 9 11  0]]


Now that we have our matrix we will be generating our QUBO expression.
A QUBO (quadratic unconstrained binary optimization) is a quadratic polynomial expression of binary variables, it is used to represent different combinatorial problems.

The goal is to find the set of binary values that minimize the expression.

For example if we have the expression : $2*x0 - 3*x1\\$
Then the minimum value will be -3 with x0 = 0 and x1 = 1.


For each path between two nodes we need to define a QUBO monomial.

In [25]:
from mpqp.execution.vqa.qubo import Qubo

def generate_qubo_expression(adj_matrix : npt.NDArray[np.int64]) -> Qubo :
    length = len(adj_matrix)
    result = 0
    for i in range(length):
        for j in range(length):
            if adj_matrix[i][j] == 0:
                continue
            current = adj_matrix[i][j] * Qubo(f'x{i},{j}')
            if isinstance(result,int):
                result = current
            else:
                result += current 
    return result

expr = generate_qubo_expression(adj_matrix)

expr.pprint()
initial_coeffs = expr.get_coeffs([])
print(initial_coeffs)

3*x0,1+4*x0,2+2*x1,0+5*x1,2+9*x2,0+11*x2,1
[(3, ['x0,1']), (4, ['x0,2']), (2, ['x1,0']), (5, ['x1,2']), (9, ['x2,0']), (11, ['x2,1'])]


Now that we have our Qubo expression we need to implement the constraints of this problem, constraints here 

The first one is the fact that we have to go through each cities exactly once, in this case we express it as xor operators.

In [18]:
def add_constraints(expr):
    for i in range(3):
        if i != 0:
            current = Qubo(f'x{i},0')
        else:
            current = Qubo(f'x{i},1')
        for j in range(1,3):
            if i == j or (i == 0 and j == 1):
                continue
            path = Qubo(f'x{i},{j}')
            print(f'{current.value} ^ {path.value}')
            expr += -20* (current ^ path)
    for i in range(3):
        if i != 0:
            current = Qubo(f'x0,{i}')
        else:
            current = Qubo(f'x1,{i}')
        for j in range(1,3):
            if i == j or (i == 0 and j == 1):
                continue
            path = Qubo(f'x{j},{i}')
            print(f'{current.value} ^ {path.value}')
            expr += -20* (current ^ path)
    return expr

expr = add_constraints(expr)


x0,1 ^ x0,2
x1,0 ^ x1,2
x2,0 ^ x2,1
x1,0 ^ x2,0
x0,1 ^ x2,1
x0,2 ^ x1,2


In [19]:
optimizers = ['Nelder-Mead', 'Powell']

In [26]:
def test_results(res, coeffs):
    current_answer = 0
    other = 0
    for i in range(len(res)):
        if res[i] == '0':
            other += coeffs[i][0]
        else:
            current_answer += coeffs[i][0]
    return current_answer <= other


In [29]:
from mpqp.execution.vqa.qaoa import MixerType, qaoa_solver

for _ in range(20):
    print(test_results(qaoa_solver(expr, 3,MixerType.MIXER_X, 'Nelder-Mead'), initial_coeffs))

True
True
True
True
True


KeyboardInterrupt: 

In [30]:
x = Qubo("x")
y = Qubo("y")
z = Qubo("z")
expr = x*3 - 2*y + 100*(x&y)
print(qaoa_solver(expr, 3,MixerType.MIXER_X,"Powell"))

01
